# Ninefold Tide — Verified Blender 4.5.15 CPU cache (private Kaggle)

Download the OFFICIAL Blender 4.5.15 LTS Linux release (~378 MB) once to the cloud; cross-check the Blender.org SHA-256, perform an internal headless .blend-save and 64px CPU render test, and retain the compressed archive privately in this notebook's output for reuse by the camera-locked v09 scene builder. This does **not** use any GPU quota and does **not** transfer private Drive character source or touch existing notebooks. All art remains unapproved.


In [ ]:
from pathlib import Path, PurePosixPath
import datetime, hashlib, io, json, os, re, subprocess, tarfile, time, urllib.request

archive_name="blender-4.5.15-linux-x64.tar.xz"
download_url="https://download.blender.org/release/Blender4.5/"+archive_name
checksum_url="https://download.blender.org/release/Blender4.5/blender-4.5.15.sha256"
work=Path("/kaggle/working")
temp=Path("/kaggle/temp")/"ninefold_verified_blender_4_5"
temp.mkdir(parents=True,exist_ok=True)
out=work/"ninefold_blender_cached"
out.mkdir(parents=True,exist_ok=True)
archive=work/archive_name
status={
 "project":"The Ninefold Tide",
 "purpose":"Download and cache one verified official Blender executable as PRIVATE Kaggle output; NO personal art or voice sources",
 "free_cpu_only":True,
 "sources_downloaded_from":["download.blender.org"],
 "blender_release":"4.5.15 LTS",
 "estimated_package_megabytes":378,
 "source_v09r1_private_drive":"https://drive.google.com/file/d/1fOMgzjCH2CYGj_9KeveiQQdPeR1vF-g_/view",
 "downloaded_archive_bytes":0,
 "checksum_official_verified":False,
 "actual_blender_version":None,
 "grease_pencil_v3_supported":False,
 "saved_blend_file":False,
 "rendered_smoke_png":False,
 "status":"STARTED",
 "no_existing_kaggle_notebooks_modified":True,
 "artistic_approval":False,
 "started_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
 "errors":[]
}
def save():
    (out/"blender_cache_status.json").write_text(json.dumps(status,indent=2)+"\n",encoding="utf8")
def fail(msg):
    status["errors"].append(str(msg)[:700])
    print("BLENDER_CACHE_ERROR",str(msg)[:700],flush=True)
save()
try:
    request=urllib.request.Request(checksum_url,headers={"User-Agent":"Ninefold-Verified-Portable-Blender/1.0"})
    with urllib.request.urlopen(request,timeout=90) as response:
        checksums=response.read(180000).decode("utf8","replace")
    line=next((x for x in checksums.splitlines() if archive_name in x),None)
    if not line: raise RuntimeError("Official checksum manifest does not name exact Blender archive")
    match=re.match(r"^([0-9a-fA-F]{64})\b",line.strip())
    if not match:raise RuntimeError("Invalid SHA256 format from official Blender checksum manifest")
    expected=match.group(1).lower()
    status["official_sha256"]=expected
    if not archive.exists():
        hasher=hashlib.sha256()
        request=urllib.request.Request(download_url,headers={"User-Agent":"Ninefold-Verified-Portable-Blender/1.0"})
        with urllib.request.urlopen(request,timeout=110) as remote,open(archive,"wb") as local:
            declared=remote.headers.get("Content-Length")
            if declared and int(declared)>575_000_000:
                raise RuntimeError("Blender download unexpectedly large")
            consumed=0
            while True:
                chunk=remote.read(2*1024*1024)
                if not chunk:break
                consumed+=len(chunk)
                if consumed>575_000_000:
                    raise RuntimeError("Blender download exceeded safe limit")
                local.write(chunk)
                hasher.update(chunk)
        actual=hasher.hexdigest()
    else:
        hasher=hashlib.sha256()
        with open(archive,"rb") as fp:
            for chunk in iter(lambda:fp.read(2*1024*1024),b""):
                hasher.update(chunk)
        actual=hasher.hexdigest()
    status["downloaded_archive_bytes"]=archive.stat().st_size
    status["downloaded_sha256"]=actual
    if actual!=expected:
        archive.unlink(missing_ok=True)
        raise RuntimeError("Official Blender package SHA256 did not match; do not execute")
    status["checksum_official_verified"]=True
    save()
    print("BLENDER_OFFICIAL_SHA256_PASS",archive.stat().st_size,flush=True)
    install=temp/"portable_blender"
    install.mkdir(exist_ok=True,parents=True)
    with tarfile.open(archive,"r:xz") as tar:
        for entry in tar.getmembers():
            n=PurePosixPath(entry.name)
            if n.is_absolute() or ".." in n.parts:
                raise RuntimeError("Official Blender package has unsafe archive path")
        tar.extractall(install,filter="data")
    binaries=[p for p in install.rglob("blender") if p.is_file()]
    if not binaries: raise RuntimeError("Official portable archive has no Blender executable")
    binary=binaries[0]
    binary.chmod(binary.stat().st_mode|0o111)
    check=subprocess.run([str(binary),"--version"],capture_output=True,text=True,timeout=70)
    if check.returncode: raise RuntimeError("Verified Blender cannot start headless on Kaggle CPU")
    status["actual_blender_version"]=(check.stdout or "").splitlines()[0]
    print("BLENDER_VERIFIED_HEADLESS",status["actual_blender_version"],flush=True)
    smoke=out/"smoke_check.py"
    smoke.write_text(
       "import bpy,json\nfrom pathlib import Path\n"
       "p=Path('/kaggle/working/ninefold_blender_cached')\n"
       "has_gp=hasattr(bpy.data,'grease_pencils_v3') or hasattr(bpy.types,'GreasePencilv3')\n"
       "scene=bpy.context.scene\n"
       "scene.render.engine='BLENDER_WORKBENCH'\n"
       "scene.render.resolution_x=64;scene.render.resolution_y=64;scene.render.resolution_percentage=100\n"
       "scene.render.filepath=str(p/'smoke_cpu_64.png')\n"
       "scene.render.image_settings.file_format='PNG'\n"
       "bpy.ops.wm.save_as_mainfile(filepath=str(p/'smoke_source.blend'))\n"
       "bpy.ops.render.render(write_still=True)\n"
       "(p/'actual_bpy_properties.json').write_text(json.dumps({'version':bpy.app.version_string,'gp_v3':has_gp,'save':(p/'smoke_source.blend').exists(),'render':(p/'smoke_cpu_64.png').exists()},indent=2))\n",
       encoding="utf8")
    env=os.environ.copy()
    env["LIBGL_ALWAYS_SOFTWARE"]="1"
    result=subprocess.run([str(binary),"--background","--factory-startup","--python",str(smoke)],
                          env=env,capture_output=True,text=True,timeout=170)
    (out/"blender_console_excerpt.txt").write_text(
        (result.stdout or "")[-4500:]+"\nSTDERR\n"+(result.stderr or "")[-3000:],encoding="utf8")
    if result.returncode:raise RuntimeError("Blender 4.5 headless smoke run failed; check console and graphics libraries")
    actual=json.loads((out/"actual_bpy_properties.json").read_text(encoding="utf8"))
    status["grease_pencil_v3_supported"]=bool(actual["gp_v3"])
    status["saved_blend_file"]=bool(actual["save"])
    status["rendered_smoke_png"]=bool(actual["render"])
    if not all((status["grease_pencil_v3_supported"],status["saved_blend_file"],status["rendered_smoke_png"])):
        raise RuntimeError("Blender executable works but the Grease Pencil/save/render preflight failed")
    status["status"]="VERIFIED_PRIVATE_BLENDER_4_5_CACHE_READY"
except Exception as exc:
    fail(type(exc).__name__+": "+str(exc))
    status["status"]="FAILED_DO_NOT_REUSE"
finally:
    status["ended_utc"]=datetime.datetime.now(datetime.timezone.utc).isoformat()
    save()
    print("NINEFOLD_BLENDER_CACHE_FINAL",json.dumps({
      "status":status["status"],
      "official_sha256_verified":status["checksum_official_verified"],
      "blender_version":status["actual_blender_version"],
      "gp_v3":status["grease_pencil_v3_supported"],
      "smoke_png":status["rendered_smoke_png"],
      "package_bytes":status["downloaded_archive_bytes"],
      "errors":status["errors"]
    },indent=2),flush=True)
